# A real, trainable CNN in 6 cells (PyTorch, Colab GPU)

This trains an actual convolutional net on MNIST digits and shows what it learned. Runtime → Change runtime type → GPU, then Run all. Should take under a minute total.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
tfm = transforms.ToTensor()
train_ds = datasets.MNIST(root='.', train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST(root='.', train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

print(f'{len(train_ds)} train images, {len(test_ds)} test images, shape {train_ds[0][0].shape}')

## The model: 2 conv layers -> pool -> 2 conv layers -> pool -> dense

`Conv2d(in_channels, out_channels, kernel_size)` is the whole idea from the theory: `out_channels` learned filters, each `kernel_size x kernel_size` and as deep as `in_channels`. `MaxPool2d(2)` halves the spatial size.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)   # 1 input channel (grayscale) -> 16 filters
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 16 -> 32 filters
        self.pool  = nn.MaxPool2d(2)
        self.fc    = nn.Linear(32 * 7 * 7, 10)        # 28x28 -> pool -> 14x14 -> pool -> 7x7

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (B,16,28,28) -> (B,16,14,14)
        x = self.pool(F.relu(self.conv2(x)))   # (B,32,14,14) -> (B,32,7,7)
        x = x.flatten(1)
        return self.fc(x)                      # 10 class scores

model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

## Train it (2 epochs is enough to hit ~98% on MNIST)

In [ ]:
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f'epoch {epoch+1}: test accuracy = {100*correct/total:.2f}%')

## See what it actually learned

The 16 filters of `conv1` are each a 3x3 window over the raw pixels — plot them directly. Then run one real test image through the trained net and look at the resulting feature maps: each channel should visibly react to different strokes of the digit (edges, loops, corners).

In [ ]:
filters = model.conv1.weight.detach().cpu().squeeze(1)   # (16, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i], cmap='gray')
    ax.axis('off')
fig.suptitle('all 16 learned conv1 filters (3x3 each)')
plt.show()

In [ ]:
img, label = test_ds[0]
model.eval()
with torch.no_grad():
    fmap1 = F.relu(model.conv1(img.unsqueeze(0).to(device)))[0].cpu()   # (16, 28, 28)

fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(img.squeeze(), cmap='gray'); axes[0].set_title(f'input ({label})'); axes[0].axis('off')
for i in range(8):
    axes[i+1].imshow(fmap1[i], cmap='gray'); axes[i+1].set_title(f'filter {i}'); axes[i+1].axis('off')
plt.tight_layout()
plt.show()